<a href="https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umairhussainn/ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Scoring, built on top of a classification proxy. Lane 2's output is a ranked review queue, not
a single yes/no per page — so the end task is scoring/ranking. But the score itself comes from
a binary classifier's predicted probability (declining vs. not), the same way the starter
notebooks used predict_proba() to rank pages, not just to label them. So: classification model
underneath, scoring/ranking task on top — the reviewer never sees "class 1", they see an
ordered list.

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/umairhussainn/ml-internship"
REPO_DIR = "ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} pages loaded")

30,000 pages loaded


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Starting proxy (what I have now): is_declining_label = trend_direction == "down" — a bucket
calculated from the current window, not a real future outcome. I'm using it honestly as a
beginner proxy, per the lane guide's own warning.

Where I'm heading (stronger target for later weeks): a future-window label — features from the
prior 90 days predicting decline over the next 30 days. That's an observed future outcome, not
a same-window bucket, so it will be a real proxy for "did this page actually get worse,"
not "does it currently look bad by one definition."

In [ ]:
print(df["trend_direction"].value_counts())
print()


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64



In [ ]:
print(df["is_declining_label"].value_counts(normalize=True).rename("share"))


is_declining_label
1    0.542067
0    0.457933
Name: share, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages the ranking puts first, what fraction are actually declining
(or, later, actually decline in the next window). I'm picking this over plain accuracy because
the real decision is capacity-limited: a reviewer only opens the top of the list, so what
matters is whether THAT top slice is trustworthy, not how the model scores on the whole 30,000
pages. This is also directly comparable to what I already measured in ML-01: hand rule
Precision@50 = 0.240, random forest = 0.740 — the number I'm trying to beat honestly, on a
proper client-holdout split, not in-sample.

In [ ]:
# Success metric.
print("Baseline (hand rule) Precision@50: 0.240")
print("Random forest Precision@50:        0.740")

Baseline (hand rule) Precision@50: 0.240
Random forest Precision@50:        0.740


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id), scoped to one client (client_id), as of the current
90-day window. This is the grain the whole lane operates on — not one row per client, not one
row per day.

In [ ]:
cols = ["content_id", "client_id", "content_type", "word_count", "impressions_90d",
        "clicks_90d", "avg_position", "ctr", "days_since_last_update",
        "content_age_days", "trend_direction", "is_declining_label"]
df[cols].head(5)

,content_id,client_id,content_type,word_count,impressions_90d,clicks_90d,avg_position,ctr,days_since_last_update,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3221.0,3803,29,10.6,0.76,20,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,2481.0,15320,7,20.3,0.05,25,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,3515.0,12581,11,36.5,0.09,20,141,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,NaN,11751,58,6.2,0.49,22,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,2803.0,19140,24,44.0,0.13,14,263,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Two pieces of real evidence from this data:

1. No single signal is a strong predictor alone. Correlating each candidate feature against
   is_declining_label one at a time, every one sits under |0.2| — days_since_last_update=0.081,
   content_age_days=-0.164, word_count=0.090, ctr=-0.062, avg_position=-0.029,
   impressions_90d=-0.018. None of these, alone, is an obvious if-statement.

2. A tight hand rule misses almost everything. The starter "stale AND visible" rule
   (days_since_last_update>=180 AND impressions_90d>=500) only flags 17 pages out of 30,000 —
   and even among those 17, it only catches 16 of the 16,262 actually-declining pages (0.1%).
   A fixed AND-rule with hard thresholds is too rigid: it needs every condition to be true at
   once, so it misses every page where the signal is spread across several moderately-weak
   features instead of concentrated in a few extreme ones. That's exactly what a model that
   weighs and combines many weak features (like a decision tree or random forest) is built to
   find, and it's exactly what the ML-01 numbers showed happening (0.240 vs 0.740 Precision@50).

In [ ]:
for col in ["days_since_last_update", "impressions_90d", "avg_position",
            "ctr", "content_age_days", "word_count"]:
    print(f"corr({col}, is_declining_label) = {df[col].corr(df['is_declining_label']):.3f}")

hand_flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
caught = df.loc[hand_flag, "is_declining_label"].sum()
total_declining = df["is_declining_label"].sum()
print(f"\nHand rule flags: {hand_flag.sum()} pages")
print(f"Catches {caught} of {total_declining} actual decliners ({caught/total_declining:.1%})")

corr(days_since_last_update, is_declining_label) = 0.081
corr(impressions_90d, is_declining_label) = -0.018
corr(avg_position, is_declining_label) = -0.029
corr(ctr, is_declining_label) = -0.062
corr(content_age_days, is_declining_label) = -0.164
corr(word_count, is_declining_label) = 0.090

Hand rule flags: 17 pages
Catches 16 of 16262 actual decliners (0.1%)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.